# ATiG 2026: LT-FH exercise using ltpred

## What you will learn

The [ltpred tutorial](https://bvilhjal.github.io/ltpred/tutorial/) builds a simulated register whose genetic liability is known. This exercise measures the shortcuts that tutorial names. No real data are used.

In class (Parts A and B, one hour):

- **A.** A wrong h² rescales the score. The ranking barely moves.
- **B.** The score is a posterior mean, not a risk. Correlation with liability is not accuracy.

Homework: **C** the score against a 0/1 family-history indicator; **D** h² from parent–offspring pairs, then a case-cohort; **E** observed-scale versus liability-scale h².

Write what you expect before each part. The product is the Q15 note.

LT-FH++ ([Pedersen et al. 2022](https://doi.org/10.1016/j.ajhg.2022.01.009)) is LT-FH ([Hujoel et al. 2020](https://doi.org/10.1038/s41588-020-0613-6)) with age of onset. ADuLT ([Pedersen et al. 2023](https://doi.org/10.1038/s41467-023-41210-z)) drops the family history. Scores use the Pearson–Aitken engine of PA-FGRS ([Krebs et al. 2024](https://doi.org/10.1016/j.ajhg.2024.09.009); [2026](https://doi.org/10.1016/j.ajhg.2025.11.016)). Model: [algorithm page](https://bvilhjal.github.io/ltpred/algorithm/).


## Setup

Install Git, Python, NumPy, SciPy and ltpred from the [setup page](https://bvilhjal.github.io/ATIG_2026/setup.html) (macOS, Windows, or Anaconda). Then run the cell below. Part 0 takes about a minute and needs under 1 GB of memory. Executed with ltpred 0.7.0 (commit `2d85ce9`); the printouts are identical under NumPy 1.26 to 2.5.


In [1]:
import numpy as np
from scipy.stats import norm, rankdata

import ltpred
print("ltpred", ltpred.__version__)

ltpred 0.7.0


## Part 0. Rebuild the tutorial register, then a larger one

The first three cells are tutorial steps 1, 3 and 4 with the tutorial's seeds, so
`cohort`, `scores`, `est`, `predicted` and `at_risk` are the tutorial's names and
the checkpoints match its printouts. Part A uses this register.

The tutorial register has 81 diagnoses, and 71 of them fall after age 40. That
is enough to see an effect and not enough to measure one, so the fourth cell
builds a five-times larger register, `big`, and scores it the same two ways.
Parts B to E use `big`.

In [2]:
from ltpred import simulate_pedigree, simulate_register_liabilities

H2 = 0.5                                      # liability-scale heritability
CIP_K, CIP_MID, CIP_SLOPE = 0.10, 60.0, 1.0 / 8.0   # logistic onset curve: 10% lifetime, half of it by age 60
EVAL_AGE = 70.0                               # everyone is followed to here
INDEX_AGE = 40.0                              # the prospective cut in step 4

# the generating cumulative-incidence curve, on a 1-year age grid
AGE_GRID = np.arange(0, 121, 1.0)
TRUE_CIP = CIP_K / (1.0 + np.exp((CIP_MID - AGE_GRID) * CIP_SLOPE))

ids, father, mother = simulate_pedigree(
    np.random.default_rng(20260921), n_founder_pairs=100, gens=2)

cohort = simulate_register_liabilities(
    np.random.default_rng(20260921), ids, father, mother,
    h2=H2, cip_ages=AGE_GRID, cip_values=TRUE_CIP, eval_age=EVAL_AGE)

print(f"{len(ids)} people, {int(cohort.status.sum())} diagnosed by age "
      f"{EVAL_AGE:.0f} ({cohort.status.mean():.1%})")
print(f"var(true genetic liability) = {cohort.genetic.var():.4f}  (target {H2})")

984 people, 81 diagnosed by age 70 (8.2%)
var(true genetic liability) = 0.5083  (target 0.5)


**Checkpoint.** 984 people, 81 diagnosed, variance 0.5083: tutorial step 1.
`cohort.genetic` is the truth that Part A compares against.

In [3]:
from ltpred import estimate_liabilities

scores = estimate_liabilities(
    cohort.ids, cohort.father, cohort.mother,
    probands=cohort.ids,
    status=cohort.status.astype(int), age=cohort.age,
    use="gwas",                          # the proband's own diagnosis is used
    cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K,
    h2=H2)

est = np.asarray(scores.est)
print(f"corr(score, true genetic liability) = {np.corrcoef(est, cohort.genetic)[0, 1]:.4f}")
print(f"score sd = {est.std():.4f}   mean posterior variance = {np.asarray(scores.var).mean():.4f}")

corr(score, true genetic liability) = 0.5502
score sd = 0.3898   mean posterior variance = 0.3558


In [4]:
at_risk = cohort.onset > INDEX_AGE          # still undiagnosed at the cut
index_time = cohort.birth_time + INDEX_AGE
probands_at_risk = [p for p, keep in zip(cohort.ids, at_risk) if keep]

predicted = estimate_liabilities(
    cohort.ids, cohort.father, cohort.mother,
    probands=probands_at_risk,
    status=cohort.status.astype(int), age=cohort.age,
    use="prediction",                    # the proband's own diagnosis is hidden
    cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K, h2=H2,
    birth_time=cohort.birth_time, index_time=index_time[at_risk])

pred_est = np.asarray(predicted.est)
print(f"{int(at_risk.sum())} of {len(ids)} probands are disease-free at age {INDEX_AGE:.0f}")
print(f"corr(score, truth) = {np.corrcoef(pred_est, cohort.genetic[at_risk])[0, 1]:.4f}"
      f"   score sd = {pred_est.std():.4f}")

974 of 984 probands are disease-free at age 40
corr(score, truth) = 0.3069   score sd = 0.2134


**Checkpoint.** Correlations 0.5502 and 0.3069: tutorial steps 3 and 4. Step 3
also noted that 0.390² + 0.356 = 0.508, the variance of `cohort.genetic`: the
variance of the scores plus the mean posterior variance recovers Var(g). Part A
asks what becomes of that identity when h² is wrong.

In [5]:
big_ids, big_father, big_mother = simulate_pedigree(
    np.random.default_rng(1), n_founder_pairs=500, gens=2)
big = simulate_register_liabilities(
    np.random.default_rng(1), big_ids, big_father, big_mother,
    h2=H2, cip_ages=AGE_GRID, cip_values=TRUE_CIP, eval_age=EVAL_AGE)


def score_big(use, h2=H2):
    """Score `big` as tutorial step 3 (use="gwas") or step 4 (use="prediction")."""
    prospective = use == "prediction"
    keep = big.onset > INDEX_AGE if prospective else np.ones(len(big.ids), bool)
    return estimate_liabilities(
        big.ids, big.father, big.mother,
        probands=[p for p, k in zip(big.ids, keep) if k],
        status=big.status.astype(int), age=big.age, use=use,
        cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K, h2=h2,
        birth_time=big.birth_time if prospective else None,
        index_time=(big.birth_time + INDEX_AGE)[keep] if prospective else None)


big_gwas = score_big("gwas")                 # every person, own diagnosis used
big_pred = score_big("prediction")           # disease-free at 40, own diagnosis hidden
big_est = np.asarray(big_gwas.est)
big_at_risk = big.onset > INDEX_AGE
big_m, big_v = np.asarray(big_pred.est), np.asarray(big_pred.var)

print(f"{len(big.ids)} people, {int(big.status.sum())} diagnosed by age 70 "
      f"({big.status.mean():.1%}); var(genetic) = {big.genetic.var():.4f}")
print(f"use='gwas':       corr(score, truth) = {np.corrcoef(big_est, big.genetic)[0, 1]:.4f}")
print(f"use='prediction': corr(score, truth) = "
      f"{np.corrcoef(big_m, big.genetic[big_at_risk])[0, 1]:.4f} on "
      f"{int(big_at_risk.sum())} probands disease-free at 40")

5075 people, 366 diagnosed by age 70 (7.2%); var(genetic) = 0.4932
use='gwas':       corr(score, truth) = 0.5222
use='prediction': corr(score, truth) = 0.2781 on 5037 probands disease-free at 40


**Checkpoint.** 5,075 people and 366 diagnoses. The two correlations, 0.52 and
0.28, are the tutorial's 0.55 and 0.31 on a register with the same generating
model but five times the people; the difference is sampling noise and the
pedigree mix, not a change in the method.

## Part A. What does a wrong h² do?

Step 3 passed the true `h2=0.5`. In a real analysis h² is an external estimate for
the disease on the liability scale, and it has no default: the
[method guide](https://bvilhjal.github.io/ltpred/guide/) says to prefer an external estimate or to cross-check it.
Here you can pass a wrong value on purpose and compare with the truth. Part A
uses the tutorial register, `cohort`.

### Q1: If you tell the scorer h² = 0.8 when the truth is 0.5, do the scores spread more or less? Does their ranking change?

**Before running:** write your expectation down with one sentence of reasoning.
The scorer's model says that a fraction h² of the liability variance is genetic
and shared with relatives.

### Q2: Score the register under h² = 0.2, 0.5 and 0.8 and tabulate the consequences.

For each assumed value, score `cohort` with `use="gwas"` exactly as in Part 0 and
print five numbers: the correlation with `cohort.genetic`; the variance of the
scores; the mean posterior variance `s.var`; their sum; and the slope of
`cohort.genetic` regressed on the score, `np.polyfit(e, cohort.genetic, 1)[0]`.

In [ ]:
print(f"{'assumed h2':>10} {'corr':>7} {'var(score)':>11} {'mean var':>9} {'sum':>7} {'slope':>7}")
for h2_assumed in (0.2, 0.5, 0.8):
    s = estimate_liabilities(
        cohort.ids, cohort.father, cohort.mother, probands=cohort.ids,
        status=cohort.status.astype(int), age=cohort.age, use="gwas",
        cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K,
        h2=...)                                # <- the assumed value
    e, v = np.asarray(s.est), np.asarray(s.var)
    corr = ...                                 # correlation with cohort.genetic
    slope = ...                                # np.polyfit(e, cohort.genetic, 1)[0]
    print(f"{h2_assumed:>10.1f} {corr:>7.4f} {e.var():>11.4f} {v.mean():>9.4f} "
          f"{e.var() + v.mean():>7.4f} {slope:>7.3f}")

**Checkpoint.** The row for 0.5 reproduces Part 0. The `sum` column should come
out close to the h² you passed in, whichever it was, and one column should barely
move at all.

### Q3: Which columns moved and which did not? What does a slope of 1 mean, and why does only the right h² give it?

A posterior mean is *calibrated* when the truth regressed on it has slope 1: among
probands scored 0.3, the true liability averages 0.3. Relate your slopes to the
guide's advice on h². Optional: scatter `cohort.genetic` against the score for
each h² and draw the fitted line.

## Part B. Who becomes a case between 40 and 70?

![Liability thresholds at 40 and 70. Higher liability is earlier onset. Prospective probands are still left of T40; an incident case has crossed T70.](figures/thresholds.png)

Step 4's correlation, 0.31, is not the accuracy of predicting a later case. Measure it on `big`. Probands in `big_pred` are still left of T40. An incident case is one of them who has crossed T70 by age 70.


### Q4: How many probands become incident cases, and what is the incidence?

Hint: `big.status` means "diagnosed by 70" and `big_at_risk` means "undiagnosed at
40". Combine them, then index with `big_at_risk` so that the result lines up with
`big_m`.

In [ ]:
y = ...          # boolean, one per proband in big_pred: diagnosed after 40 and by 70
print(f"{int(y.sum())} incident cases among {len(y)} probands ({y.mean():.1%})")

**Checkpoint.** A few hundred incident cases, an incidence a little below the
register's 7.2% case share, and a count of probands equal to the register minus
the people diagnosed before 40.

### Q5: Compute the AUC for three predictors of that outcome on the same probands.

The AUC is the probability that a random incident case scores above a random
non-case. With ranks from `scipy.stats.rankdata` it is the Mann–Whitney statistic:
if `r` are the ranks of the predictor, `n1` the number of cases and `n0` the number
of non-cases, AUC = (sum of the cases' ranks − n1(n1 + 1)/2) / (n1 · n0).

The predictors: the prospective score `big_m`; the true genetic liability
`big.genetic[big_at_risk]`, which is the ceiling for any predictor built from
genetics alone; and the `use="gwas"` score `big_est` restricted to the same
people, which used each proband's own diagnosis.

**Before running:** order the three AUCs.

In [ ]:
def auc(x, y):
    r = rankdata(x)
    n1, n0 = y.sum(), (~y).sum()
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

print(f"AUC prospective score          = {auc(big_m, y):.3f}")
print(f"AUC true genetic liability     = {auc(..., y):.3f}")
print(f"AUC use='gwas' score, same people = {auc(..., y):.3f}")

**Checkpoint.** The `use="gwas"` score should sit near 1. If it does not, you have
not restricted it to the probands in `big_pred`. The other two should be clearly
above 0.5 and clearly apart.

### Q6: Why is the ceiling below 1 although it is the truth? Where does the gap between the prospective score and the ceiling come from? What does the third AUC say about `use="gwas"` in a prospective claim?

On `big`, print `big_m.var()`, `big_est.var()` and `np.asarray(big_gwas.var).mean()`, next to `big.genetic.var()`. The retrospective score saw each proband's own diagnosis. The prospective score did not.


### Q7: Turn the score into a risk and check it against the observed incidence.

`big_m` is not a probability. Given the relatives, full liability is Normal with mean `m = big_m` and variance `s² = big_v + 1 − h²`, because `use="prediction"` left the proband's own status out. With `T_a = Φ⁻¹(1 − CIP(a))` as in the figure,

risk = [Φ((T40 − m)/s) − Φ((T70 − m)/s)] / Φ((T40 − m)/s).

Compute it for every proband. Compare the mean with the Q4 incidence, then within fifths of `big_m`.


In [ ]:
m, v = big_m, big_v
T40 = norm.ppf(1.0 - np.interp(INDEX_AGE, AGE_GRID, TRUE_CIP))
T70 = norm.ppf(1.0 - np.interp(EVAL_AGE, AGE_GRID, TRUE_CIP))
s = np.sqrt(v + 1.0 - H2)
p_free_at_40 = norm.cdf((T40 - m) / s)
risk = ...                                    # the formula above

print(f"mean model risk {risk.mean():.4f}   observed incidence {y.mean():.4f}")

fifths = np.array_split(np.argsort(m, kind="stable"), 5)
for label, g in zip(("lowest", "second", "middle", "fourth", "highest"), fifths):
    print(f"{label:>8} fifth  n={len(g)}  events={int(y[g].sum()):3d}  "
          f"observed {y[g].mean():.3f}  model risk {risk[g].mean():.3f}")

**Checkpoint.** Mean risk within about half a percentage point of the incidence. The extreme fifths should separate. Neighbouring middle fifths have a few dozen events and need not. Why does the risk need `big_v`?


## Part C. What the score buys over a 0/1 indicator (homework)

The usual alternative to a family-history score is the crude indicator "any
first-degree relative diagnosed", the covariate a GWAS might adjust for. This part
puts it next to the score on `big`, both retrospectively and prospectively.

### Q8: How much does the score improve on a 0/1 family-history indicator?

For every person in `big`, compute (i) their own status, (ii) a 0/1 indicator of
any diagnosed first-degree relative (a parent or a full sibling, not the proband),
and (iii) the count of diagnosed first-degree relatives, all with follow-up to 70.
Compare their R² against `big.genetic` with that of the `use="gwas"` score
`big_est`, which also saw everything to age 70.

Then compare prospectively. The fair opponent of `big_m` is an indicator that
knows only what was known at the proband's 40th birthday, so recompute (ii) and
(iii) counting a relative only if `big.birth_time + big.onset` is at or before the
proband's index time. Report their AUCs for the Part B outcome. Why is own status
missing from the prospective comparison?

In [ ]:
big_idx = {p: i for i, p in enumerate(big.ids)}
sibs = {}                                     # children grouped by (father, mother)
for i, (fa, mo) in enumerate(zip(big.father, big.mother)):
    if fa in big_idx and mo in big_idx:
        sibs.setdefault((fa, mo), []).append(i)


def first_degree(i):
    """Row indices of person i's parents and full siblings present in the register."""
    fa, mo = big.father[i], big.mother[i]
    rels = [big_idx[p] for p in (fa, mo) if p in big_idx]
    if fa in big_idx and mo in big_idx:
        rels += [j for j in sibs[(fa, mo)] if j != i]
    return np.asarray(rels, int)


relatives = [first_degree(i) for i in range(len(big.ids))]
big_index_time = big.birth_time + INDEX_AGE

n_fdr = np.array([... for r in relatives], float)             # diagnosed by 70
n_fdr_40 = np.array([... for i, r in enumerate(relatives)], float)   # diagnosed by the proband's index time
fh01, fh01_40 = (n_fdr > 0).astype(float), (n_fdr_40 > 0).astype(float)

for name, x in (("own status", big.status.astype(float)), ("FH indicator 0/1", fh01),
                ("# affected FDRs", n_fdr), ("use='gwas' score", big_est)):
    print(f"R2  {name:18s} {...:.3f}")                        # squared correlation with big.genetic
for name, x in (("FH indicator 0/1 at 40", fh01_40[big_at_risk]),
                ("# affected FDRs at 40", n_fdr_40[big_at_risk]),
                ("prospective score", big_m)):
    print(f"AUC {name:22s} {auc(x, y):.3f}")

**Checkpoint.** Retrospectively the score's R² should beat own status and the indicator. Prospectively the AUC gap should be much smaller. Fewer people are FH-positive at 40 than at 70.


## Part D. Where does h² come from? (homework)

Part A showed that h² is not a nuisance parameter. The [method guide](https://bvilhjal.github.io/ltpred/guide/)
suggests a cross-check: h² is about twice a first-degree tetrachoric correlation,
and `ltpred.tetrachoric` estimates one from the 2×2 table of two binary statuses.
This part runs the cross-check on both registers, asks why it works, and then
breaks it with a realistic sampling design.

### Q9: Estimate h² from parent–offspring pairs in the tutorial register, then in `big`.

Pair every person with each parent whose id is in the table (both parents give a
pair) and call `tetrachoric(parent_status, child_status)`. Report the 2×2 table
and 2ρ ± 2 SE for `cohort` and for `big`.

In [ ]:
from ltpred.tetrachoric import tetrachoric, tetrachoric_table


def parent_offspring_status(reg):
    idx = {p: i for i, p in enumerate(reg.ids)}
    parent, child = [], []
    for kid, fa, mo in zip(reg.ids, reg.father, reg.mother):
        for par in (fa, mo):
            if par in idx:
                parent.append(...); child.append(...)   # statuses of the pair

    return np.asarray(parent), np.asarray(child)


def table(parent, child):
    """Counts: both diagnosed, parent only, child only, neither."""
    return np.array([(parent & child).sum(), (parent & ~child).sum(),
                     (~parent & child).sum(), (~parent & ~child).sum()])


for label, reg in (("tutorial register", cohort), ("larger register", big)):
    par, kid = parent_offspring_status(reg)
    t = tetrachoric(par, kid)
    print(f"{label}: table {table(par, kid)}  ->  h2 = {2 * t.rho:+.2f} +/- {2 * t.se:.2f}")

**Checkpoint.** Both estimates should bracket 0.5, the second with about half the
first's standard error, and the first cell of each table (both diagnosed) should be
a small number.

### Q10: Why does twice the tetrachoric work here, and what would break it in a real register?

Consider what the tetrachoric assumes about the threshold, what this simulation
assumes about shared environment, and how the register was sampled.

### Q11: Run the cross-check on an iPSYCH-style case-cohort.

Real registers are often case-enriched. iPSYCH, for instance, holds every person
diagnosed with one of its disorders in its birth cohorts plus a random sample of
the rest, so the sample's case share far exceeds the prevalence. From `big`, keep
every diagnosed person and each undiagnosed person with probability 0.2 (seed
20260924). Report the sample's case rate against the population's, then estimate
h² from its parent–offspring pairs as in Q9.

Then undo the design. Each kept pair represents 1/(π_parent · π_child) pairs of
the population, where π is 1 for a case and 0.2 for a control. Sum those weights
per cell of the 2×2 table, compare the result with `big`'s table from Q9, and
run `tetrachoric_table` on the rounded weighted counts.

In [ ]:
Q_KEEP = 0.2
keep = big.status | (np.random.default_rng(20260924).random(len(big.ids)) < Q_KEEP)
pi = np.where(big.status, 1.0, Q_KEEP)                  # inclusion probability

par, kid, w = [], [], []
for i in np.flatnonzero(keep):
    for p in (big.father[i], big.mother[i]):
        if p in big_idx and keep[big_idx[p]]:
            ...                                          # statuses of the pair, and its weight
par, kid, w = np.asarray(par), np.asarray(kid), np.asarray(w)

t = tetrachoric(par, kid)
print(f"kept {int(keep.sum())} of {len(big.ids)}: case rate {...:.1%} vs {big.status.mean():.1%}")
print(f"{len(par)} pairs, table {table(par, kid)}  ->  naive h2 = {2 * t.rho:+.2f} +/- {2 * t.se:.2f}")

weighted = ...                                           # weight sums per cell, in table() order
tw = tetrachoric_table(*np.round(weighted).astype(int))
print(f"weighted table {np.round(weighted).astype(int)}  ->  h2 = {2 * tw.rho:+.2f}")

**Checkpoint.** About a quarter of the register is kept and its case rate is
roughly four times the population's. The naive estimate should be well above 0.5;
the weighted table should look like `big`'s own table from Q9, and its estimate
should be back near 0.5.

## Part E. Which scale is h² on? (homework)

![A liability split into a 0/1 outcome. The right tail is the cases, 10% when K = 0.10. Everyone on one side receives the same y.](figures/observed_scale.png)

GREML and LD-score regression report h² on the 0/1 scale. Oversampling cases makes that number depend on the sample case proportion P. With prevalence K and z = φ(Φ⁻¹(1 − K)) ([Lee et al. 2011](https://doi.org/10.1016/j.ajhg.2011.02.002); [2012](https://doi.org/10.1002/gepi.21614)),

h²_liab = h²_obs · K²(1 − K)² / (z² · P(1 − P)).

In a population sample, P = K. `observed_to_liability_h2` and `liability_to_observed_h2` are this map and its inverse. The same P(1 − P) sets Neff = 4 N₁ N₀ / N.

K here is the lifetime prevalence 0.10 given to the scorer. "Diagnosed by 70" has prevalence about 0.078. An h² for that phenotype has to use 0.078.


### Q12: What observed-scale h² would each design report?

The truth is h² = 0.5 with K = 0.10. Use `liability_to_observed_h2` to compute
the h² each design would report: (i) a population sample (`prop_cases=None`),
(ii) a 1:1 case/control study (P = 0.5), and (iii) a 1:4 study (P = 0.2). Convert
each back with `observed_to_liability_h2` and check that you recover 0.5.

In [ ]:
from ltpred import observed_to_liability_h2, liability_to_observed_h2

for name, P in (("population sample", None), ("1:1 case/control", 0.5), ("1:4 case/control", 0.2)):
    obs = ...                            # liability_to_observed_h2(H2, CIP_K, P)
    back = ...                           # observed_to_liability_h2(obs, CIP_K, P)
    print(f"{name:18s} P={CIP_K if P is None else P:.2f}  h2_obs={float(obs):.4f}  round trip={float(back):.4f}")

**Checkpoint.** All three round-trip to 0.5 exactly, and the three observed-scale
values differ from each other by more than a factor of two.

### Q13: A GWAS on a population sample reports h² = 0.17. Feed it to the scorer unconverted.

Score `cohort` exactly as in Part A, but at the observed-scale value from Q12 (i)
instead of a liability-scale h². Report the correlation with `cohort.genetic` and
the calibration slope. Which Part A row does this imitate, and why is no error
raised? Then convert the 1:1 and the 1:4 designs' values from Q12 while forgetting
the P factor. What does the scorer do with each result?

In [ ]:
obs_population = float(liability_to_observed_h2(H2, CIP_K, None))
s = estimate_liabilities(
    cohort.ids, cohort.father, cohort.mother, probands=cohort.ids,
    status=cohort.status.astype(int), age=cohort.age, use="gwas",
    cip_ages=AGE_GRID, cip_values=TRUE_CIP, k_pop=CIP_K,
    h2=...)                              # the unconverted observed-scale value
e = np.asarray(s.est)
print(f"h2 = {obs_population:.4f}: corr = {...:.4f}  slope = {...:.3f}")

for P in (0.5, 0.2):
    forgotten = float(observed_to_liability_h2(liability_to_observed_h2(H2, CIP_K, P), CIP_K))
    print(f"P = {P}: forgetting the P factor gives h2 = {forgotten:.2f}")
    ...                                  # try scoring cohort at `forgotten`; catch ValueError

**Checkpoint.** The correlation stays near Part A's, the slope is well above 2, and
of the two forgotten-factor values one is refused and one is silently accepted.

### Q14: How much information does each design carry?

Compute Neff = 4 N₁ N₀ / N for `big` as a population sample (366 cases among
5,075), a 1:1 study of 2,500 cases and 2,500 controls, a biobank GWAS with 5,000
cases and 45,000 controls, and a consortium with 50,000 cases and 150,000
controls. Express Neff/N through P alone. Why do case/control GWAS oversample
cases, and why did Part D's tetrachoric never need P while the Lee transformation
does?

In [ ]:
for name, n_case, n_ctrl in (("big, population", int(big.status.sum()), len(big.ids) - int(big.status.sum())),
                             ("1:1 study", 2_500, 2_500),
                             ("biobank 1:9", 5_000, 45_000),
                             ("consortium 1:3", 50_000, 150_000)):
    n = n_case + n_ctrl
    neff = ...                           # 4 * n_case * n_ctrl / n
    print(f"{name:16s} N={n:7d}  P={n_case / n:.3f}  Neff={neff:8.0f}  Neff/N={neff / n:.3f}")

**Checkpoint.** Neff/N depends on P alone and equals 1 only for the 1:1 study;
`big` carries the information of far fewer than 5,075 people.

### Q15: Analysis note.

In at most 150 words, state what the family-history score estimates, what it
showed in this exercise, and what it cannot establish. Cite one number from each
of Parts A and B and, if you did them, Parts C to E.

---
Part of [ATIG 2026](https://github.com/bvilhjal/ATIG_2026), teaching day
[24 September](https://github.com/bvilhjal/ATIG_2026/tree/main/teaching_days/2026-09-24).